## import libraries

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.pipeline import Pipeline

## load datasets

In [2]:
train_df = pd.read_csv('data/6/train.csv')
valid_df = pd.read_csv('data/6/valid.csv')
# test_df = pd.read_csv('data/6/test.csv')

train_df.dropna(subset=['text', 'sentiment'], inplace=True)
valid_df.dropna(subset=['text', 'sentiment'], inplace=True)

## **Model Development** using Train set

- Vecotorizing (Preprocessing + Tokenization)

- Hyperparamete tuning using CV on Train set

- Re-train best model on Train set

In [12]:
results_df = pd.DataFrame(columns=["vectorizer", "valid_accuracy", "valid_F1_score"])

def evaluation_results(vectorizer_name, grid_search, train_df, valid_df, train_predictions, valid_predictions, results_df=results_df):
    # Best parameters
    print("Best parameters:")
    print(grid_search.best_params_)

    # Evaluation on train_df
    print("\nEvaluation on train_df:")
    print("Train Accuracy:", round(accuracy_score(train_df['sentiment'], train_predictions),2), '%')
    print("Train F1-macro:", round(f1_score(train_df['sentiment'], train_predictions, average='macro'),2), '%')
    print("\nClassification report on train_df:")
    print(classification_report(train_df['sentiment'], train_predictions))

    # Evaluation on valid_df
    print("\nEvaluation on valid_df:")
    print("Validation Accuracy:", round(accuracy_score(valid_df['sentiment'], valid_predictions),2), '%')
    print("Validation F1-macro:", round(f1_score(valid_df['sentiment'], valid_predictions, average='macro'),2), '%')
    print("\nClassification report on valid_df:")
    print(classification_report(valid_df['sentiment'], valid_predictions))

    # Append results to results_df
    results = {
        "vectorizer": vectorizer_name,
        "valid_accuracy": round(accuracy_score(valid_df['sentiment'], valid_predictions),2),
        "valid_F1_score": round(f1_score(valid_df['sentiment'], valid_predictions, average='macro'),2),
    }

    results_df = pd.concat([results_df, pd.DataFrame([results])], ignore_index=True)
    return results_df


### TF-IDF representation

Hyperparameter tuning using GridsearchCV

In [ ]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True)
tfidf_model = LogisticRegression(max_iter=1000)

tfidf_pipeline = Pipeline([
    ('vectorizer', tfidf_vectorizer),
    ('classifier', tfidf_model)
])

tfidf_param_grid = {
    'vectorizer__ngram_range': [(1, 2)],
    'vectorizer__min_df': [2, 5, 10],
    'vectorizer__max_df': [0.9],
    'vectorizer__max_features': [5000, 10000],
    'vectorizer__sublinear_tf': [True],
    'classifier__C': [0.001, 0.01, 0.1, 1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

tfidf_grid_search = GridSearchCV(
    estimator=tfidf_pipeline,
    param_grid=tfidf_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

tfidf_grid_search.fit(train_df['text'], train_df['sentiment'])

tfidf_best_pipeline = tfidf_grid_search.best_estimator_
tfidf_pred_train = tfidf_best_pipeline.predict(train_df['text'])
tfidf_pred_valid = tfidf_best_pipeline.predict(valid_df['text'])

results_df = evaluation_results('TF-IDF', tfidf_grid_search, train_df, valid_df, tfidf_pred_train, tfidf_pred_valid, results_df)

Best parameters:
{'classifier__C': 1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2), 'vectorizer__sublinear_tf': True}

Evaluation on train_df:
Train Accuracy: 0.92 %
Train F1-macro: 0.89 %

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.73      0.97      0.83      1001
    positive       0.99      0.91      0.95      3911

    accuracy                           0.92      4912
   macro avg       0.86      0.94      0.89      4912
weighted avg       0.94      0.92      0.92      4912


Evaluation on valid_df:
Validation Accuracy: 0.88 %
Validation F1-macro: 0.83 %

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.65      0.85      0.74       143
    positive       0.96      0.88      0.9

### BoW representation

In [ ]:
bow_vectorizer = CountVectorizer(lowercase=True)
bow_model = LogisticRegression(max_iter=1000)

bow_pipeline = Pipeline([
    ('vectorizer', bow_vectorizer),
    ('classifier', bow_model)
])

bow_param_grid = {
    'vectorizer__ngram_range': [(1, 1), (1, 2)],
    'vectorizer__min_df': [2, 5, 10],
    'vectorizer__max_df': [0.9, 1.0],
    'vectorizer__max_features': [3000, 5000, 10000],
    'vectorizer__binary': [False, True],
    'classifier__C': [0.001, 0.01, 0.1, 1],
    'classifier__penalty': ['l2'],
    'classifier__solver': ['liblinear'],
    'classifier__class_weight': ['balanced']
}

bow_grid_search = GridSearchCV(
    estimator=bow_pipeline,
    param_grid=bow_param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1
)

bow_grid_search.fit(train_df['text'], train_df['sentiment'])

bow_best_pipeline = bow_grid_search.best_estimator_
bow_pred_train = bow_best_pipeline.predict(train_df['text'])
bow_pred_valid = bow_best_pipeline.predict(valid_df['text'])

results_df = evaluation_results('Bag of Words', bow_grid_search, train_df, valid_df, bow_pred_train, bow_pred_valid, results_df)

Best parameters:
{'classifier__C': 0.1, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l2', 'classifier__solver': 'liblinear', 'vectorizer__binary': False, 'vectorizer__max_df': 0.9, 'vectorizer__max_features': 10000, 'vectorizer__min_df': 2, 'vectorizer__ngram_range': (1, 2)}

Evaluation on train_df:
Train Accuracy: 0.95 %
Train F1-macro: 0.93 %

Classification report on train_df:
              precision    recall  f1-score   support

    negative       0.84      0.95      0.89      1001
    positive       0.99      0.95      0.97      3911

    accuracy                           0.95      4912
   macro avg       0.92      0.95      0.93      4912
weighted avg       0.96      0.95      0.95      4912


Evaluation on valid_df:
Validation Accuracy: 0.9 %
Validation F1-macro: 0.84 %

Classification report on valid_df:
              precision    recall  f1-score   support

    negative       0.73      0.78      0.75       143
    positive       0.94      0.92      0.93   

### Word2vec representation

## **Model selection** using Validation set

## **Final Performance** using Test set